# SEEP frequency-state recall — CUB200, seed 3407

Enable **GPU + Internet**. Add Input `trungkiennguyn/lgsp-dataset` and
**your original SEEP task1..task10 checkpoints from the same run**.
Attach Inputs before background Save Version. The supplied metrics ZIP
contains no checkpoints. Reeval runs first; missing inputs never trigger training.
No CLS Steering / N1 / CCSR / Gaussian alignment / transport is used.
Local validation is synthetic; this notebook has no precomputed CUB result.

In [ ]:
import sys, os, json, hashlib, base64, subprocess, shutil, time, traceback
from pathlib import Path
RUN_MODE = 'reeval'  # Alternative: 'train_collect' starts a NEW full 10 x 30 run.
CHECKPOINT_ROOT = '/kaggle/input'
CHECKPOINT_MAP = None  # JSON path or dict {"1": "/kaggle/input/my-run/task1_checkpoint.pth", ... "10": ...}
TRUSTED_CHECKPOINTS = False  # True ONLY for your own trusted Namespace/pickle checkpoints.
DATA_ROOT = None  # Set official CUB root here if discovery finds duplicates.
REPO = Path('/kaggle/working/SEEP_Frequency_State')
READY = Path('/kaggle/working/cub200_ready_frequency_state')
OUTPUT = Path('/kaggle/working/seep_frequency_state_seed3407')
NPZ_PATH = Path('/kaggle/working/pretrained/ViT-B_16.npz')
assert RUN_MODE in ('reeval', 'train_collect')
assert not OUTPUT.exists(), 'Use a fresh OUTPUT directory to preserve previous results.'
OUTPUT.mkdir(parents=True)
(OUTPUT/'run_status.json').write_text(json.dumps({'status':'configured','full_protocol_complete':False}))
NOTEBOOK_STARTED = time.perf_counter()
def record_cell_failure(result):
    error = result.error_before_exec or result.error_in_exec
    if error is None: return
    (OUTPUT/'run_status.json').write_text(json.dumps({'status':'failed','error':str(error),'full_protocol_complete':False}))
    (OUTPUT/'timing.json').write_text(json.dumps({'notebook_elapsed_seconds':time.perf_counter()-NOTEBOOK_STARTED}))
    with (OUTPUT/'run.log').open('a',encoding='utf8') as stream:
        traceback.print_exception(type(error),error,error.__traceback__,file=stream)
    summary_path = OUTPUT/'summary.json'
    if summary_path.exists():
        summary=json.loads(summary_path.read_text()); summary['full_protocol_complete']=False
        summary['process_success']=False; summary_path.write_text(json.dumps(summary,indent=2))
    # Works even when clone/dependency setup fails before project imports.
    import zipfile
    archive=OUTPUT.parent/'SEEP_FREQUENCY_STATE_CUB200_metrics.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix.lower() in {'.json','.csv','.txt','.log','.patch','.md','.ipynb','.xml'}:
                bundle.write(path,str(path.relative_to(OUTPUT)))
    print('Failed/partial archive:',archive)
if 'get_ipython' in globals():
    get_ipython().events.register('post_run_cell',record_cell_failure)
files = sorted(p for p in Path(CHECKPOINT_ROOT).rglob('*')
               if p.is_file() and p.suffix.lower() in {'.pth','.pt','.ckpt'})
print('Checkpoint candidate files (not yet loaded/validated):', len(files))
for p in files[:40]: print(f'{p.stat().st_size/2**20:.1f} MiB  {p}')

In [ ]:
BASE_COMMIT = '80081081ab42769d102f83d447705da0639e2fb1'
PATCH_SHA256 = '8f274c9b85868d105eb6ef614ae445ab3d0321a358c84f6893d4f6f049c5aa14'
PATCH_B64 = '...'
DELIVERY = {'base_commit': '80081081ab42769d102f83d447705da0639e2fb1', 'branch': 'feat/task-frequency-state', 'patch_sha256': '8f274c9b85868d105eb6ef614ae445ab3d0321a358c84f6893d4f6f049c5aa14', 'source_files_sha256': {'vision_transformer.py': '26a78b4d8653d0e03af9d99c5b34a4067c61bc0c7d00b8d8f448c3b167e725a4', 'engine.py': '01ac70aae604fb356ea058bb5b878137170990c912779c95ccae3c18c6c674ff', 'datasets.py': 'd9a75e0685cf6f2b95a6af57f4be854c12e647e9e71bafed30a95e06eb43f835', 'continual_datasets/continual_datasets.py': '89d9ed00a4c273ef131bbdef79ec394fd903e20ed1ce62b1c8c36aecf9b7c8d1', 'modules/frequency_state.py': '9d49ac3879ec4145ef720c133422cd9135780ab6879aa181ee22b9245a7fa8ef', 'tools/frequency_setup.py': '60df75fcdc4716772bd0c2f623dedbe858b8ab599ebcc338c9c199f553df33d4', 'tools/frequency_run.py': '12425cb9b76eecd01ada91735bb086b83fbb1123c283b8399c41ce2ff7da4fd7', 'tools/prepare_cub200.py': '5ec826cd800bc9a2038c9dd432c5b589d2f5696c8470a47acbdac5d53bad1ee2', 'tools/build_frequency_delivery.py': '6836288ff3c5035a64249007602540299f4bd7441944cc63ac21676731384493', 'tools/verify_frequency_delivery.py': 'adc493026cf0632a2bc0731fd135fcd691fef494a7a3fb463874d7cbac29dd7d', 'tests/test_frequency_state.py': '5e1409d79573feca54ba65680936257d373e1536b513d1779f4e2578d933bb2e', 'README_FREQUENCY_STATE.md': '3737443b5e959f55723651b39b684786a0ae356f1716f79107740663ff78b978', 'references/align_only_seed3407/provenance.json': 'ceff45e6a12f3d778ec827858bb5027181ec110edc3672ba65a1f5442247a654', 'references/align_only_seed3407/official_split_manifest.json': 'ccf1a0227ee8f3e72bebb73fce7896b79ea48b784c1a44770a8a5f78cfcd391f', 'references/align_only_seed3407/source_manifest.json': '8703961a833ed7c1a77a909ef88485aa5dde191b6575efb350160607c182491c'}, 'serialization': 'LF', 'source_mode': 'embedded patch; no remote feature branch required', 'reused_dgca_patch_sha256': 'ecec522a42518819d2504890b6c7b6fede6938c5aca7d5d47322c2cd6cc9996e'}
assert not REPO.exists(), 'Use a new REPO directory; existing work is preserved.'
subprocess.check_call(['git','clone','https://github.com/thanhphu25/SEEP.git',str(REPO)])
subprocess.check_call(['git','config','core.autocrlf','false'], cwd=REPO)
subprocess.check_call(['git','checkout','--detach',BASE_COMMIT], cwd=REPO)
patch = base64.b64decode(PATCH_B64)
assert hashlib.sha256(patch).hexdigest() == PATCH_SHA256
patch_path = REPO/'implementation.patch'
patch_path.write_bytes(patch)
subprocess.check_call(['git','apply','--check',str(patch_path)], cwd=REPO)
subprocess.check_call(['git','apply',str(patch_path)], cwd=REPO)
for name, expected in DELIVERY['source_files_sha256'].items():
    assert hashlib.sha256((REPO/name).read_bytes()).hexdigest() == expected, name
(OUTPUT/'source_manifest.json').write_text(json.dumps(DELIVERY,indent=2))
shutil.copyfile(patch_path, OUTPUT/'implementation.patch')
shutil.copyfile(REPO/'README_FREQUENCY_STATE.md',OUTPUT/'README.md')
sys.path.insert(0,str(REPO))
from tools.frequency_setup import export
def checked_command(command, phase):
    tick = time.perf_counter()
    with (OUTPUT/'run.log').open('a',encoding='utf8') as log:
        log.write('\nPHASE '+phase+'\n'); log.flush()
        process = subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,encoding='utf8',errors='replace',env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONIOENCODING':'utf8'})
        try:
            for line in process.stdout: print(line,end=''); log.write(line); log.flush()
            result = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    (OUTPUT/(phase+'_result.json')).write_text(json.dumps({'returncode':result,'seconds':time.perf_counter()-tick}))
    if result != 0:
        (OUTPUT/'run_status.json').write_text(json.dumps({'status':'failed','phase':phase,'returncode':result,'full_protocol_complete':False}))
        summary_path = OUTPUT/'summary.json'
        if summary_path.exists():
            summary=json.loads(summary_path.read_text()); summary['full_protocol_complete']=False
            summary_path.write_text(json.dumps(summary,indent=2))
        print('Partial/failure archive:',export(OUTPUT))
        raise subprocess.CalledProcessError(result,command)
print('Pinned source and patch verified:',PATCH_SHA256)
checked_command([sys.executable,'tools/build_frequency_delivery.py'],'notebook_export')
shutil.copyfile(REPO/'SEEP_Frequency_State_CUB200_Kaggle.ipynb',OUTPUT/'SEEP_Frequency_State_CUB200_Kaggle.ipynb')

In [ ]:
# Keep installed torch/torchvision; timm shim is idempotent on Python 3.13.
checked_command([sys.executable,'-c',
    "from tools.frequency_setup import dependencies; dependencies("+repr(str(OUTPUT))+")"], 'dependencies')
import torch, torchvision
print('Actual stack:',sys.version,torch.__version__,torchvision.__version__)
assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings.'
print('GPU:',torch.cuda.get_device_name())

In [ ]:
# Tests use synthetic CPU tensors; no scientific accuracy claim.
checked_command([sys.executable,'-m','pytest','-q','tests/test_frequency_state.py',
                 '-p','no:cacheprovider','--junitxml='+str(OUTPUT/'tests.xml')], 'tests')

In [ ]:
CONFIG = dict(run_mode=RUN_MODE,checkpoint_root=CHECKPOINT_ROOT,checkpoint_map=CHECKPOINT_MAP,
    trusted_checkpoints=TRUSTED_CHECKPOINTS,data_path=str(READY),output=str(OUTPUT),
    npz_path=str(NPZ_PATH),device='cuda:0',num_workers=4)
CONFIG_PATH = OUTPUT/'notebook_config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
# Inventory and extraction happen BEFORE expensive evaluation. No images or optimizer step.
checked_command([sys.executable,'tools/frequency_run.py','--config',str(CONFIG_PATH),'--inventory-only'], 'inventory')
report=json.loads((OUTPUT/'checkpoint_inventory.json').read_text())
for row in report.get('boundaries',[]):
    print(row['boundary'],row['status'],'epoch=',row.get('epoch'),'path=',row['path'])
print('Missing:',report.get('missing_boundaries',[]))

In [ ]:
# Official metadata + split hashes, never a random train/test split.
checked_command([sys.executable,'-c',
    "from tools.frequency_setup import prepare_data; import json; print(json.dumps(prepare_data("
    +repr(None if DATA_ROOT is None else str(DATA_ROOT))+","+repr(str(READY))+"),indent=2))"], 'data')
shutil.copyfile(READY/'official_split_manifest.json',OUTPUT/'official_split_manifest.json')

In [ ]:
# Full preflight verifies saved args, frozen routing NPZ, strict dynamic load, transforms and GPU.
# The runner performs that preflight before evaluating any boundary. No optimizer in reeval.
# Progress includes boundary/task/images/elapsed/ETA; all stdout is retained in run.log.
checked_command([sys.executable,'tools/frequency_run.py','--config',str(CONFIG_PATH)], 'evaluation')

In [ ]:
# Recompute from exported CSV matrices and require successful process exit.
import numpy as np
from modules.frequency_state import CHANNELS
from tools.frequency_run import write_metrics
status=json.loads((OUTPUT/'run_status.json').read_text())
result=json.loads((OUTPUT/'evaluation_result.json').read_text())
matrices={k:np.loadtxt(OUTPUT/f'task_accuracy_matrix_{k}.csv',delimiter=',') for k in CHANNELS}
checks=json.loads((OUTPUT/'mutation_checks.json').read_text())
summary=write_metrics(OUTPUT,matrices,process_success=result['returncode']==0,
    checks_pass=len(checks)==10 and all(c['passed'] for c in checks))
assert status['status']=='complete' and summary['full_protocol_complete']
archive=export(OUTPUT)
print(json.dumps({'primary_delta_pp':summary['snapshot_minus_live_percentage_points']['raw_exact'],
                  'screening_signal':summary['screening_signal'],'full_protocol_complete':True},indent=2))
from IPython.display import display, FileLink
print('Metrics ZIP (no model checkpoints):',archive)
display(FileLink(str(archive)))